# Preprocessing Decisions and Model Assumptions

**Topic:** Data Preprocessing

In [ ]:
import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, VBox, HTML
from IPython.display import display, clear_output

from tkh_utils import PALETTE, FONT, base_layout

---
## What you'll explore

By the end of this notebook you will be able to:

- **Describe** how an algorithm's core properties determine which preprocessing steps it needs
- **Explain** why there is no universal preprocessing checklist. The right steps depend on the algorithm
- **Interpret** the decision table to quickly identify which steps are required, recommended, or unnecessary for a given algorithm family

> **Tip:** Step through each algorithm family. Notice that the first family, the one that assumes a linear relationship, requires the most preprocessing. That is the family your first algorithm belongs to.

---
## How we got here

This folder has covered every major preprocessing technique: imputation, encoding, scaling, train-test split, imbalance handling, leakage prevention, and saving clean data. Throughout, we flagged which steps were model-dependent.

This final notebook brings those flags together into a single framework. The question is not "what preprocessing should I always do?" It is "what does my algorithm assume about the data, and how do I meet those assumptions?"

---
## Why this matters for data science

Two data scientists can work with the same Titanic dataset and build very different preprocessing pipelines, both correct, because they are targeting different algorithms. Understanding why each step is needed, not just that it is a "best practice," is what lets you make those decisions confidently.

---
## Three algorithm families

Every algorithm you will encounter in this course belongs to one of three families, defined by how they use feature values:

| Family | Core mechanic | Key question asked |
|---|---|---|
| **Assumes a linear relationship** | Finds the best-fit line or plane | "What is the linear combination of features that best predicts the outcome?" |
| **Measures distances** | Finds the most similar training examples | "Which training points are closest to this new point?" |
| **Splits on thresholds** | Asks yes/no questions about features | "Is feature X above or below value T?" |

Each family makes different mathematical assumptions about the data, and those assumptions determine the preprocessing it needs.

---
## Try it yourself: which preprocessing does each family need?

In [ ]:
out = Output()

FAMILIES = {
    "Algorithms that assume a linear relationship": {
        "short": "Linear relationship",
        "description": (
            "These algorithms find the best straight line (or hyperplane) through the data. "
            "They assume that increasing a feature by one unit always changes the outcome "
            "by the same fixed amount. "
            "<strong>Your first algorithm, linear regression, falls in this family.</strong>"
        ),
        "coming_up": "You will meet linear regression in the upcoming weeks.",
        "steps": [
            {
                "name": "Handle missing values",
                "status": "required",
                "reason": (
                    "These algorithms cannot compute with NaN inputs. "
                    "A single missing value causes a crash or silently wrong predictions."
                ),
            },
            {
                "name": "Encode categorical columns",
                "status": "required",
                "reason": (
                    "These algorithms do math on features. "
                    "One-hot encoding is the safe default: it avoids implying a false numeric ordering."
                ),
            },
            {
                "name": "Feature scaling",
                "status": "required",
                "reason": (
                    "These algorithms optimize by adjusting weights iteratively. "
                    "When features are on very different scales, "
                    "the optimizer takes uneven steps and converges slowly, or not at all."
                ),
            },
            {
                "name": "Address skew (e.g. log-transform fare)",
                "status": "recommended",
                "reason": (
                    "Linear algorithms perform best when the relationship between features "
                    "and outcome is actually linear. Extreme skew can distort that relationship. "
                    "A log transform often helps but is not always required."
                ),
            },
            {
                "name": "Handle class imbalance",
                "status": "recommended",
                "reason": (
                    "Most linear classifiers support a class_weight parameter. "
                    "For regression tasks, imbalance is typically not a concern."
                ),
            },
        ],
    },
    "Algorithms that measure distances": {
        "short": "Distance-based",
        "description": (
            "These algorithms classify or predict by looking at which training examples "
            "are most similar to a new data point, measured by distance. "
            "Two data points are 'close' if their feature values are numerically similar."
        ),
        "coming_up": "You will encounter this family when you explore similarity-based methods.",
        "steps": [
            {
                "name": "Handle missing values",
                "status": "required",
                "reason": (
                    "Distance calculations require complete numeric values. "
                    "NaN cannot be subtracted from another number."
                ),
            },
            {
                "name": "Encode categorical columns",
                "status": "required",
                "reason": (
                    "One-hot encoding is especially important here. "
                    "Ordinal encoding would imply that 'Southampton = 2' is "
                    "twice as far from 'Cherbourg = 0' as from 'Queenstown = 1', which is meaningless."
                ),
            },
            {
                "name": "Feature scaling",
                "status": "required",
                "reason": (
                    "This is the most critical step for distance-based algorithms. "
                    "A feature with a large range (fare: 0–512) will dominate the distance "
                    "calculation over a small-range feature (age: 0–80), "
                    "regardless of which is more informative."
                ),
            },
            {
                "name": "Address skew (e.g. log-transform fare)",
                "status": "helpful",
                "reason": (
                    "Extreme outliers can distort distance calculations "
                    "even after scaling. A log transform reduces the influence of extreme values "
                    "before the scaler is applied."
                ),
            },
            {
                "name": "Handle class imbalance",
                "status": "helpful",
                "reason": (
                    "Distance-based algorithms do not have a built-in class_weight parameter. "
                    "Resampling (oversampling or SMOTE) is the typical approach."
                ),
            },
        ],
    },
    "Algorithms that split on thresholds": {
        "short": "Threshold-based",
        "description": (
            "These algorithms make decisions by asking a sequence of yes/no questions about "
            "individual features: 'Is fare > 50? Is age < 30?' "
            "Each split divides the data into two groups. "
            "The threshold position adjusts automatically regardless of the feature's scale."
        ),
        "coming_up": "You will meet this family in the supervised learning folder.",
        "steps": [
            {
                "name": "Handle missing values",
                "status": "required",
                "reason": (
                    "Most sklearn tree implementations require finite values. "
                    "A small number of modern variants (HistGradientBoosting) handle NaN natively, "
                    "but imputation is the safe default."
                ),
            },
            {
                "name": "Encode categorical columns",
                "status": "required",
                "reason": (
                    "Standard sklearn tree implementations require numeric input. "
                    "Ordinal encoding is often sufficient: trees find their own meaningful thresholds. "
                    "Some non-sklearn libraries (e.g. CatBoost) handle raw categories natively."
                ),
            },
            {
                "name": "Feature scaling",
                "status": "not_needed",
                "reason": (
                    "Threshold splits are completely scale-invariant. "
                    "Whether fare is 100 dollars or 0.8 on a scaled axis, "
                    "the tree adjusts its split point accordingly. "
                    "Scaling adds no value here."
                ),
            },
            {
                "name": "Address skew (e.g. log-transform fare)",
                "status": "not_needed",
                "reason": (
                    "Skew does not affect where a tree places its split points. "
                    "The tree finds the threshold that best separates the classes "
                    "regardless of the distribution shape."
                ),
            },
            {
                "name": "Handle class imbalance",
                "status": "recommended",
                "reason": (
                    "Most tree implementations support class_weight='balanced'. "
                    "This is often the most convenient approach. "
                    "no resampling needed, just pass a parameter."
                ),
            },
        ],
    },
}

STATUS_CONFIG = {
    "required":    {"label": "Required",     "bg": "#EBFBEE", "border": "#2F9E44", "text": "#2F9E44"},
    "recommended": {"label": "Recommended",  "bg": "#FFF3BF", "border": "#E67700", "text": "#E67700"},
    "helpful":     {"label": "Helpful",      "bg": "#E7F5FF", "border": "#1971C2", "text": "#1971C2"},
    "not_needed":  {"label": "Not needed",   "bg": "#F1F3F5", "border": "#ADB5BD", "text": "#868E96"},
}

family_dd = Dropdown(
    options=list(FAMILIES.keys()),
    value=list(FAMILIES.keys())[0],
    description="Algorithm family:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="600px"),
)

def render(change=None):
    family = FAMILIES[family_dd.value]

    header_html = (
        f'<div style="font-family:Inter,Arial,sans-serif;max-width:640px;margin-bottom:12px;">'
        f'<div style="border-left:4px solid {PALETTE["primary"]};padding:12px 16px;'
        f'background:#EEF2FF;border-radius:4px;margin-bottom:10px;">'
        f'<div style="font-size:13px;color:{PALETTE["primary"]};font-weight:600;margin-bottom:4px;">'
        f'ALGORITHM FAMILY</div>'
        f'<div style="font-size:14px;color:#212529;line-height:1.6;">{family["description"]}</div>'
        f'</div>'
        f'<div style="font-size:12px;color:{PALETTE["muted"]};font-style:italic;">'
        f'{family["coming_up"]}</div>'
        f'</div>'
    )

    steps_html = '<div style="font-family:Inter,Arial,sans-serif;max-width:640px;">'
    for step in family["steps"]:
        cfg = STATUS_CONFIG[step["status"]]
        steps_html += (
            f'<div style="border-left:4px solid {cfg["border"]};padding:10px 14px;'
            f'background:{cfg["bg"]};border-radius:4px;margin-bottom:8px;">'
            f'<div style="display:flex;align-items:center;gap:10px;margin-bottom:4px;">'
            f'<strong style="font-size:14px;">{step["name"]}</strong>'
            f'<span style="background:{cfg["border"]};color:#fff;padding:2px 8px;'
            f'border-radius:10px;font-size:11px;font-weight:600;">{cfg["label"]}</span>'
            f'</div>'
            f'<div style="font-size:13px;color:#495057;line-height:1.6;">{step["reason"]}</div>'
            f'</div>'
        )
    steps_html += '</div>'

    with out:
        clear_output(wait=True)
        display(HTML(header_html + steps_html))

family_dd.observe(render, names="value")
display(VBox([family_dd, out]))
render()

---
## Quick-reference decision table

In [ ]:
# Decision table as a Plotly figure for easy visual comparison

steps   = ["Handle missing values", "Encode categoricals",
           "Feature scaling", "Address skew", "Handle imbalance"]
families = ["Assumes linear\nrelationship", "Measures\ndistances", "Splits on\nthresholds"]

# R = Required, Rec = Recommended, H = Helpful, N = Not needed
table = [
    ["Required", "Required", "Required"],
    ["Required", "Required (OHE)", "Required"],
    ["Required", "Required", "Not needed"],
    ["Recommended", "Helpful", "Not needed"],
    ["Recommended", "Helpful", "Recommended"],
]

colors_map = {
    "Required":          "#EBFBEE",
    "Required (OHE)":    "#EBFBEE",
    "Recommended":       "#FFF3BF",
    "Helpful":           "#E7F5FF",
    "Not needed":        "#F1F3F5",
}
text_colors_map = {
    "Required":          "#2F9E44",
    "Required (OHE)":    "#2F9E44",
    "Recommended":       "#E67700",
    "Helpful":           "#1971C2",
    "Not needed":        "#868E96",
}

cell_fill  = [[colors_map[table[r][c]] for c in range(3)] for r in range(5)]
cell_font_colors = [[text_colors_map[table[r][c]] for c in range(3)] for r in range(5)]

fig = go.Figure(go.Table(
    columnwidth=[2.2, 1, 1, 1],
    header=dict(
        values=["<b>Preprocessing step</b>"] + [f"<b>{f}</b>" for f in families],
        fill_color=PALETTE["primary"],
        font=dict(color="white", size=12),
        align="center",
        height=40,
    ),
    cells=dict(
        values=[steps] + [[table[r][c] for r in range(5)] for c in range(3)],
        fill_color=[
            ["#ffffff"] * 5,
            [cell_fill[r][0] for r in range(5)],
            [cell_fill[r][1] for r in range(5)],
            [cell_fill[r][2] for r in range(5)],
        ],
        font=dict(
            size=12,
            color=[
                ["#212529"] * 5,
                [cell_font_colors[r][0] for r in range(5)],
                [cell_font_colors[r][1] for r in range(5)],
                [cell_font_colors[r][2] for r in range(5)],
            ],
        ),
        align=["left", "center", "center", "center"],
        height=36,
    ),
))

layout = base_layout(title="Preprocessing Steps vs Algorithm Family: Quick Reference")
layout.update(height=310, margin=dict(l=20, r=20, t=60, b=20))
go.FigureWidget(data=fig.data, layout=layout)

---
## What's happening?

The three families differ in one critical way: **whether scale matters**.

- If an algorithm does arithmetic across features (linear, distance-based), scale matters and scaling is required.
- If an algorithm only asks whether a single feature is above or below a threshold, scale does not matter and you can skip it.

Everything else (imputation, encoding) is required by almost all sklearn algorithms simply because they need finite numeric input. The details (which encoding method, which imputer) vary by family.

### The Titanic case

For Titanic specifically:
- `age` (20% missing) and `cabin` (77% missing) require imputation for all families
- `sex`, `embarked`, `pclass` require encoding for all families
- `fare` (skewness 4.8) requires scaling only for linear and distance-based families
- Class imbalance (38% survived) is worth addressing for all families, but the method differs

### Coming up soon

The first algorithm you will meet is **linear regression**, a member of the "assumes a linear relationship" family. After everything you have learned in this folder, you are ready. You know exactly which preprocessing steps it needs and why.

---
## Key takeaway

> **There is no universal preprocessing checklist. Start with your algorithm's assumptions, identify which steps those assumptions require, and skip the rest.**

---
*Next up: supervised/01_linear_regression.ipynb, your first algorithm*